# Surgical Instrument Detector — Dataset Exploration
Visualize dataset statistics and sample annotations before training.

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import yaml

# Adjust to your dataset
DATASET_DIR = Path('../data/endovis')  # or '../data/cholec80'
DATA_YAML = Path('../configs/data_endovis.yaml')

with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)

CLASS_NAMES = cfg['names']
print('Classes:', CLASS_NAMES)

In [ ]:
# Count images and labels per split
for split in ['train', 'val', 'test']:
    imgs = list((DATASET_DIR / split / 'images').glob('*.jpg'))
    lbls = list((DATASET_DIR / split / 'labels').glob('*.txt'))
    print(f'{split}: {len(imgs)} images, {len(lbls)} label files')

In [ ]:
# Class distribution
from collections import Counter

label_dir = DATASET_DIR / 'train' / 'labels'
class_counts = Counter()

for lbl_file in label_dir.glob('*.txt'):
    with open(lbl_file) as f:
        for line in f:
            line = line.strip()
            if line:
                cls_id = int(line.split()[0])
                class_counts[CLASS_NAMES[cls_id]] += 1

fig, ax = plt.subplots(figsize=(10, 4))
labels = list(class_counts.keys())
values = list(class_counts.values())
ax.bar(labels, values, color='steelblue')
ax.set_title('Training set — instrument instance counts')
ax.set_xlabel('Instrument')
ax.set_ylabel('Count')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Visualize random samples with bounding boxes
import random

img_dir = DATASET_DIR / 'train' / 'images'
lbl_dir = DATASET_DIR / 'train' / 'labels'

COLORS = plt.cm.get_cmap('tab10', len(CLASS_NAMES))

def draw_boxes(img_path, lbl_path):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    if lbl_path.exists():
        with open(lbl_path) as f:
            for line in f:
                parts = list(map(float, line.strip().split()))
                if len(parts) < 5:
                    continue
                cls_id = int(parts[0])
                cx, cy, bw, bh = parts[1:5]
                x1 = int((cx - bw / 2) * w)
                y1 = int((cy - bh / 2) * h)
                x2 = int((cx + bw / 2) * w)
                y2 = int((cy + bh / 2) * h)
                color = tuple(int(c * 255) for c in COLORS(cls_id)[:3])
                cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
                cv2.putText(img, CLASS_NAMES[cls_id], (x1, y1 - 5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    return img

sample_imgs = random.sample(list(img_dir.glob('*.jpg')), min(6, len(list(img_dir.glob('*.jpg')))))
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, img_path in zip(axes.flat, sample_imgs):
    lbl_path = lbl_dir / img_path.with_suffix('.txt').name
    img = draw_boxes(img_path, lbl_path)
    ax.imshow(img)
    ax.set_title(img_path.name, fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()